In [20]:
import feedparser
import re

MAX_ARTICLES_PER_JOURNAL = 5

def strip_html(text: str) -> str:
    return re.sub(r"<[^>]+>", "", text).strip()

SKIP_PREFIXES = (
    "Author Correction:",
    "Publisher Correction:",
    "Correction:",
    "Erratum:",
    "Retraction:",
)


In [21]:
def fetch_articles(
    rss_url: str,
    max_articles: int = MAX_ARTICLES_PER_JOURNAL,
    link_filter: str | None = None,
) -> list[dict]:
    feed = feedparser.parse(rss_url)
    articles = []
    for entry in feed.entries:
        title = strip_html(entry.get("title", ""))
        link = entry.get("link", "")
        if any(title.startswith(prefix) for prefix in SKIP_PREFIXES):
            continue
        if link_filter and link_filter not in link:
            continue
        articles.append({
            "title": title,
            "summary": strip_html(entry.get("summary", entry.get("description", "")))[:600],
            "link": link,
            "published": entry.get("published", ""),
        })
        if len(articles) >= max_articles:
            break
    return articles

In [22]:
import re

def fetch_articles_clean(
    rss_url: str,
    max_articles: int = MAX_ARTICLES_PER_JOURNAL,
    link_filter: str | None = None,
) -> list[dict]:
    feed = feedparser.parse(rss_url)
    articles = []
    for entry in feed.entries:
        title = strip_html(entry.get("title", ""))
        link = entry.get("link", "")
        if any(title.startswith(prefix) for prefix in SKIP_PREFIXES):
            continue
        if link_filter and link_filter not in link:
            continue
        raw_summary = strip_html(entry.get("summary", entry.get("description", "")))
        clean_summary = re.sub(
            r'^[\w\s]+,\s*Published online:[^;]+;\s*doi:\S+\s*', '', raw_summary
        ).strip()
        if not clean_summary or clean_summary == title:
            clean_summary = "(RSS 未提供摘要)"
        articles.append({
            "title": title,
            "summary": clean_summary[:800],
            "link": link,
            "published": entry.get("published", ""),
        })
        if len(articles) >= max_articles:
            break
    return articles

nature_articles = fetch_articles_clean("https://www.nature.com/nature.rss", link_filter="s41586")

for i, a in enumerate(nature_articles, 1):
    print(f"[{i}] {a['title']}")
    print(f"    摘要: {a['summary'][:120]}...")
    print(f"    連結: {a['link']}")
    print()

[1] Bohmian mechanics remains unchallenged by tunnelling experiment
    摘要: mechanics remains unchallenged by tunnelling experiment...
    連結: https://www.nature.com/articles/s41586-026-10450-6

[2] β-Arrestin condensates regulate G-protein-coupled receptor function
    摘要: multifunctional adaptor proteins that regulate G-protein-coupled receptors (GPCRs), form phase-separated condensates, su...
    連結: https://www.nature.com/articles/s41586-026-10539-y

[3] Human haematopoietic stem cells remember inflammatory stress
    摘要: xenograft inflammation–recovery models and single-cell multiomics, a haematopoietic stem cell population after inflammat...
    連結: https://www.nature.com/articles/s41586-026-10522-7

[4] Darkness and body size shaped end-Cretaceous marine extinction patterns
    摘要: ecosystem modelling shows that impact-driven darkness and body-size-dependent extinction thresholds, rather than carbon-...
    連結: https://www.nature.com/articles/s41586-026-10541-4

[5] Sparse-to-de